# T2.6 – DBRepo API Reimplementation

Loads all experiment data exclusively from the DBRepo REST API.
No local file reads are used in the final data-loading layer.

**API base URL**: `https://test.dbrepo.tuwien.ac.at`  
**Authentication**: HTTP Basic Auth (username + password)  
**Endpoints used**:
| Method | Path | Purpose |
|--------|------|---------|
| `GET` | `/api/v1/database/{db_id}/view/{view_id}/data` | Fetch view rows (paginated) |

**Views used** (defined in T2.4):
| View | Filter | Rows |
|------|--------|------|
| `ml_accident_features` | — | 8 358 (full dataset) |
| `ml_fatal_accidents` | severity = 1 | 197 |
| `ml_serious_accidents` | severity = 2 | 1 828 |
| `ml_rural_accidents` | rural_urban = 'Rural' | 5 973 |
| `ml_high_speed_accidents` | speed_limit ≥ 60 mph | 4 970 |

In [1]:
import json
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from requests.auth import HTTPBasicAuth

In [2]:
ENDPOINT  = "https://test.dbrepo.tuwien.ac.at"
USERNAME  = "Chrisvenator"
PASSWORD  = "y$cm9e$Z$Lt@cePXUuCH"
PAGE_SIZE = 1000

IDS_FILE = Path("dbrepo_ids.json")
ids      = json.loads(IDS_FILE.read_text())
DB_ID    = ids["database_id"]
AUTH     = HTTPBasicAuth(USERNAME, PASSWORD)
HEADERS  = {"Accept": "application/json"}

print("Database ID:", DB_ID)
print("Views configured:", list(ids["views"].keys()))

Database ID: f36ef3e2-1aee-4526-b3ea-82f661a9261a
Views configured: ['ml_accident_features', 'ml_fatal_accidents', 'ml_serious_accidents', 'ml_rural_accidents', 'ml_high_speed_accidents']


## 1 · DBRepo data loader

In [3]:
def fetch_view_df(view_id: str, page_size: int = PAGE_SIZE) -> pd.DataFrame:
    """Fetch all rows of a DBRepo view via paginated REST API calls.

    Raises RuntimeError on connection failure or unexpected HTTP status.
    Returns an empty DataFrame if the view contains no rows.
    """
    url = f"{ENDPOINT}/api/v1/database/{DB_ID}/view/{view_id}/data"
    pages: list[pd.DataFrame] = []
    page = 0
    while True:
        try:
            r = requests.get(
                url,
                params={"page": page, "size": page_size},
                auth=AUTH,
                headers=HEADERS,
                timeout=30,
            )
        except requests.exceptions.ConnectionError as exc:
            raise RuntimeError(f"Connection error fetching view {view_id} page {page}: {exc}") from exc
        except requests.exceptions.Timeout:
            raise RuntimeError(f"Timeout fetching view {view_id} page {page}")

        if r.status_code != 200:
            raise RuntimeError(
                f"HTTP {r.status_code} fetching view {view_id} page {page}: {r.text[:200]}"
            )

        batch = r.json()
        if not isinstance(batch, list):
            raise RuntimeError(f"Unexpected response type {type(batch).__name__} for view {view_id}")
        if not batch:
            break

        pages.append(pd.DataFrame(batch))
        if len(batch) < page_size:
            break
        page += 1

    return pd.concat(pages, ignore_index=True) if pages else pd.DataFrame()

## 2 · Load all five views

In [4]:
print("Loading views from DBRepo REST API...")
dfs: dict[str, pd.DataFrame] = {}

for view_name, view_id in ids["views"].items():
    t0 = time.time()
    dfs[view_name] = fetch_view_df(view_id)
    elapsed = time.time() - t0
    print(f"  {view_name}: {len(dfs[view_name])} rows  ({elapsed:.1f}s)")

print("\nDone. Available DataFrames:", list(dfs.keys()))

Loading views from DBRepo REST API...


  ml_accident_features: 8358 rows  (21.9s)


  ml_fatal_accidents: 197 rows  (2.9s)


  ml_serious_accidents: 1828 rows  (4.2s)


  ml_rural_accidents: 5973 rows  (13.9s)


  ml_high_speed_accidents: 4970 rows  (10.3s)

Done. Available DataFrames: ['ml_accident_features', 'ml_fatal_accidents', 'ml_serious_accidents', 'ml_rural_accidents', 'ml_high_speed_accidents']


## 3 · Verification against local-file baseline

Compares the API-loaded `ml_accident_features` view against the original raw CSV to confirm the reimplementation produces identical results.

In [5]:
LOCAL_CSV = Path("../data/raw/1-rta2009to2013fortableau.csv")
local_raw = pd.read_csv(LOCAL_CSV, low_memory=False)
print(f"Local CSV  : {len(local_raw)} rows, {len(local_raw.columns)} columns")
api_df = dfs["ml_accident_features"]
print(f"API view   : {len(api_df)} rows, {len(api_df.columns)} columns")

Local CSV  : 8358 rows, 42 columns
API view   : 8358 rows, 25 columns


In [6]:
failures: list[str] = []

def check(label: str, passed: bool, detail: str = "") -> None:
    status = "PASS" if passed else "FAIL"
    msg = f"  [{status}] {label}"
    if not passed:
        msg += f"  → {detail}"
        failures.append(label)
    print(msg)

# Row count
check("Row count",
      len(api_df) == len(local_raw),
      f"API={len(api_df)}, local={len(local_raw)}")

# police_ref universe
api_refs   = set(api_df["police_ref"].astype(str))
local_refs = set(local_raw["Police_ref"].astype(str))
check("police_ref universe",
      api_refs == local_refs,
      f"{len(api_refs - local_refs)} extra in API, {len(local_refs - api_refs)} missing")

# Numeric column sums
COL_MAP = {
    "casualties":    ("Casualties",  0.01),
    "vehicles":      ("Vehicles",    0.01),
    "speed_limit_mph": ("Speed_lim", 0.01),
}
for api_col, (local_col, tol) in COL_MAP.items():
    api_sum   = float(api_df[api_col].sum())
    local_sum = float(local_raw[local_col].sum())
    check(f"{api_col} sum",
          abs(api_sum - local_sum) <= tol,
          f"API={api_sum:.2f}, local={local_sum:.2f}")

# Severity class distribution
api_sev   = api_df["severity_id"].value_counts().sort_index().to_dict()
local_sev = local_raw["Severity"].value_counts().sort_index().to_dict()
check("severity_id distribution",
      api_sev == local_sev,
      f"API={api_sev}, local={local_sev}")

# Filtered view counts
check("ml_fatal_accidents row count",
      len(dfs["ml_fatal_accidents"]) == 197,
      f"got {len(dfs['ml_fatal_accidents'])}")
check("ml_serious_accidents row count",
      len(dfs["ml_serious_accidents"]) == 1828,
      f"got {len(dfs['ml_serious_accidents'])}")

print()
if failures:
    print(f"Verification FAILED on: {failures}")
else:
    print("Verification PASSED: API results are identical to local-file version.")

  [PASS] Row count
  [PASS] police_ref universe
  [PASS] casualties sum
  [PASS] vehicles sum
  [PASS] speed_limit_mph sum
  [PASS] severity_id distribution
  [PASS] ml_fatal_accidents row count
  [PASS] ml_serious_accidents row count

Verification PASSED: API results are identical to local-file version.


## 4 · Usage example

Downstream ML code replaces `pd.read_csv(LOCAL_CSV)` with a single call:

In [7]:
# Drop-in replacement for pd.read_csv — use this in all downstream experiment code.
def load_accident_features() -> pd.DataFrame:
    """Return the full accident feature table from DBRepo."""
    return fetch_view_df(ids["views"]["ml_accident_features"])

def load_view(view_name: str) -> pd.DataFrame:
    """Return any named view from DBRepo by its configured name."""
    if view_name not in ids["views"]:
        raise ValueError(f"Unknown view '{view_name}'. Available: {list(ids['views'].keys())}")
    return fetch_view_df(ids["views"][view_name])

# Smoke test
sample = load_accident_features()
print(f"load_accident_features(): {len(sample)} rows × {len(sample.columns)} columns")
print(sample.dtypes)

load_accident_features(): 8358 rows × 25 columns
accident_date                str
accident_time                str
area_hectares            float64
carriageway_hazard_id      int64
casualties                 int64
crossing_control_id        int64
crossing_facility_id       int64
day_of_week                int64
easting                    int64
junction_control_id        int64
junction_detail_id         int64
lad_name                     str
latitude                 float64
light_condition_id         int64
longitude                float64
northing                   int64
police_ref                 int64
road_cond_id               int64
road_type_id               int64
rural_urban                  str
severity_id                int64
special_condition_id       int64
speed_limit_mph            int64
vehicles                   int64
weather_condition_id       int64
dtype: object
